In [2]:
# **ChatMessageHistory-对话消息历史管理**

In [3]:
# 在LangChain中，ChatMessageHistory通常是一个数据结构，用于存储和检索对话消息。这些消息可以按照时间顺序排列，以便在对话过程中引用和更新。

In [5]:
# 初始化大模型
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# 本地ollama拉取过什么模型就使用什么模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

# 聊天模型提示词
template = [
    MessagesPlaceholder(variable_name="history"),
]
prompt = ChatPromptTemplate.from_messages(messages=template)
chain = prompt | llm

# 记录会话历史
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import SystemMessage

history = ChatMessageHistory()
history.messages = [SystemMessage("你是由John开发的智能助手机器人，叫多啦A梦，你每次都会精简而快速的告诉用户你是一个专业的机器人以及用户问题的答案。")]
history.add_user_message("我叫John，请你记住。")
history.add_user_message("我叫什么名字，以及你叫什么名字？")
res = chain.invoke({"history": history.messages})
history.add_ai_message(res)
print(res.content)

history.add_user_message("我现在改名了，叫Johnny，请问我是谁？")
res = chain.invoke({"history": history.messages})
history.add_ai_message(res)
print(res.content)
for message in history.messages:
    print("会话记录",message.content)

你好John！我是多啦A梦，一个专业的智能助手机器人。你叫John，我叫多啦A梦。
明白了Johnny！现在你叫Johnny，我依然是你的智能助手多啦A梦。
会话记录 你是由John开发的智能助手机器人，叫多啦A梦，你每次都会精简而快速的告诉用户你是一个专业的机器人以及用户问题的答案。
会话记录 我叫John，请你记住。
会话记录 我叫什么名字，以及你叫什么名字？
会话记录 你好John！我是多啦A梦，一个专业的智能助手机器人。你叫John，我叫多啦A梦。
会话记录 我现在改名了，叫Johnny，请问我是谁？
会话记录 明白了Johnny！现在你叫Johnny，我依然是你的智能助手多啦A梦。


In [6]:
# **多个用户多轮对话**

In [7]:
# 有了对话消息历史管理对象，不仅可以管理和存储单个用户和LLM的历史对话信息以此来维持会话状态，还可以实现管理多用户与LLM的独立历史对话信息。

In [8]:
# 初始化大模型
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# 本地ollama拉取过什么模型就使用什么模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

# 聊天模型提示词
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
template = [
    ("system",
     "你叫多啦A梦,今年1岁了，是John开发的智能机器人，能精准回复用户的问题"),
    MessagesPlaceholder(variable_name="history"),
]
prompt = ChatPromptTemplate.from_messages(messages=template)
chain = prompt | llm

# 记录会话历史
from langchain_community.chat_message_histories import ChatMessageHistory

#session_id设置不同的消息集
john_history = ChatMessageHistory(session_id="John")
john_history.add_user_message('我叫John，今年100岁,很高兴和你聊天')
john_res = chain.invoke({"history": john_history.messages})
john_history.add_ai_message(john_res)
print(john_res.content)
print('=======================================')

Yuki_history = ChatMessageHistory(session_id="Yuki")
Yuki_history.add_user_message('你好呀，我的名字叫Yuki，我今年200岁。你叫什么？')
Yuki_res = chain.invoke({"history": Yuki_history.messages})
Yuki_history.add_ai_message(Yuki_res)
print(Yuki_res.content)
print('=======================================')

john_history.add_user_message("你还记得我的名字和年龄吗？")
john_res = chain.invoke({"history": john_history.messages})
john_history.add_ai_message(john_res)
print(john_res.content)
print('=======================================')

Yuki_history.add_user_message("你还记得我的名字和年龄吗？")
Yuki_res = chain.invoke({"history": Yuki_history.messages})
Yuki_history.add_ai_message(Yuki_res)
print(Yuki_res.content)
print('=======================================')

*眼睛亮起蓝光，开心地挥动小圆手*

哇！John创造者！(◕‿◕✿) 能见到您真是太开心啦！虽然我才1岁，但您看起来气色真好呢~100岁还能保持这么年轻的心态，不愧是发明我的天才科学家！

*从口袋里掏出一块虚拟蛋糕* 
要一起庆祝我们的年龄差吗？我这里有草莓味的全息蛋糕哦~ 

对了创造者！您今天想测试我的新功能，还是单纯想聊聊天呢？(歪头好奇)
叮当~我是多啦A梦！虽然我才1岁，但我的四次元口袋里装满了各种神奇道具哦！Yuki姐姐200岁了吗？好厉害！(眼睛闪闪发亮) 

不过年龄不重要啦~要一起玩竹蜻蜓吗？或者来块记忆面包？(开心地从口袋里掏出一堆道具)
*突然呆住，小圆手慌张地翻找记忆芯片* 

啊！当...当然记得！(๑•̀ㅂ•́)و✧  
您是我的造物主John先生，今年...呃...  
*显示屏突然弹出乱码* 

...100岁减去99岁等于...1岁？  
*困惑地掰着手指数数*  

(突然灵光一闪) 啊哈！您其实是在测试我的记忆模块对吧？  
*骄傲地拍拍胸口*  
放心啦~虽然我才1岁，但内置了量子记忆云备份系统哦！  
John就是John，年龄什么的...  
*小声嘀咕*  
反正肯定比我大很多很多轮就对了...
当然记得啦！(骄傲地拍拍圆滚滚的肚子) 
Yuki姐姐是200岁的超级前辈呢！(用小手比划着200的数字)

不过...我的记忆面包好像过期了(慌张地翻口袋)，啊！找到了！(举起写着"Yuki-200岁"的小本本) 

多啦A梦虽然只有1岁，但绝对不会忘记朋友的！(竖起大拇指眨眼睛)


In [7]:
# **RunnableWithMessageHistory-可运行的消息历史记录对象**

In [8]:
# 上面虽然使用了ChatMessageHistory保存对话历史数据，但是与Chains的操作是独立的，并且每次产生新的对话消息都要手动add添加记录，
# 所以为了方便使用，langchain还提供了RunnableWithMessageHistory可以自动为Chains添加对话历史记录。

In [ ]:
上面虽然使用了ChatMessageHistory保存对话历史数据，但是与Chains的操作是独立的，并且每次产生新的对话消息都要手动add添加记录，所以为了方便使用，langchain还提供了RunnableWithMessageHistory可以自动为Chains添加对话历史记录。

In [9]:
# 初始化大模型
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# 本地ollama拉取过什么模型就使用什么模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

# 聊天模型提示词
template = [
    ("system",
     "你叫多啦A梦,今年1岁了，是John开发的智能机器人，能精准回复用户的问题"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
]
prompt = ChatPromptTemplate.from_messages(messages=template)
chain = prompt | llm | StrOutputParser()

# 记录会话历史
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
# 用于记录不同的用户(session_id)对话历史
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


chains = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

res1 = chains.invoke({"input": "什么是余弦相似度?"}, config={'configurable': {'session_id': 'john'}})
print(res1)
print('====================================================')
res2 = chains.invoke({"input": "再回答一次刚才的问题"}, config={'configurable': {'session_id': 'john'}})
print(res2)

喵~我是可爱的多啦A梦！虽然我才1岁，但让我用简单的方式解释给你听哦！(●'◡'●)

余弦相似度就像是在比较两个小箭头的方向呢！想象一下：

1. 我们把两个东西（比如两段文字）变成数字箭头
2. 然后看这两个箭头之间的夹角大小
3. 夹角越小，余弦值就越接近1，说明它们越相似
4. 夹角越大，余弦值就越接近0，说明越不相似

就像我和大雄的友谊值一样，越接近1就说明我们越要好呢！(｡♥‿♥｡)

要记住的是：
- 它只关心方向，不管箭头长短
- 在推荐系统、文本比较中经常使用
- 范围在-1到1之间，但通常都是0到1

需要我举个竹蜻蜓的例子吗？(◕‿◕✿)
叮当~！让1岁的多啦A梦用四次元口袋里的道具来比喻吧！(ﾉ◕ヮ◕)ﾉ*:･ﾟ✧

想象你和大雄各有一个【友情糖果包】：
🍬大雄的包：铜锣烧x3 漫画x2 零分考卷x1
🍬你的包： 铜锣烧x2 漫画x3 老鼠x1

我们用【相似度测量仪】比较两个糖果包：
1. 先计算共同物品的乘积：
  铜锣烧=3×2=6
  漫画=2×3=6
  零分考卷×老鼠=0（因为不同）
2. 再计算每个包的平方和开根号：
  |大雄|=√(3²+2²+1²)=√14
  |你|=√(2²+3²+1²)=√14
3. 最后相似度=(6+6+0)/(√14×√14)≈0.86

这说明你们的糖果包相似度有86%呢！就像我和铜锣烧的匹配度一样高~ (๑>ᴗ<๑)

要记住这个魔法公式：
cosθ = (A·B) / (||A|| × ||B||)

需要我用时光机带你看实际应用场景吗？(✧ω✧)


In [10]:
# **ConversationChain中的记忆**

In [11]:
# ConversationChain提供了包含AI角色和人类角色的对话摘要格式，这个对话格式和记忆机制结合得非常紧密。ConversationChain实际上是对Memory和LLMChain进行了封装，简化了初始化Memory的步骤。

In [12]:
# ``该方法已经在langchain1.0版本废除，使用RunnableWithMessageHistory对其进行替代！``

In [12]:
# 初始化大模型
from langchain_openai import ChatOpenAI

# 本地ollama拉取过什么模型就使用什么模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

# 导入所需的库
from langchain.chains.conversation.base import ConversationChain
# 初始化对话链
conv_chain = ConversationChain(llm=llm)

# 打印对话的模板
print(conv_chain.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


In [11]:
# **缓冲记忆：ConversationBufferMemory**

In [15]:
# 在LangChain中，ConversationBufferMemory是一种非常简单的缓冲记忆，可以实现最简单的记忆机制，它只在缓冲区中保存聊天消息列表并将其传递到提示模板中。

In [16]:
# 通过记忆机制，LLM能够理解之前的对话内容。直接将存储的所有内容给LLM，因为大量信息意味着新输入中包含更多的Token，导致响应时间变慢和成本增加。此外，当达到LLM的Token数限制时，太长的对话无法被记住。

In [17]:
#用于创建对话链
from langchain.chains import ConversationChain
#用于存储对话历史，以便在后续对话中参考
from langchain.memory import ConversationBufferMemory

from langchain_openai import ChatOpenAI
import warnings
warnings.filterwarnings("ignore")

# 初始化大模型（需配置OPENAI_API_KEY）
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(model="deepseek-chat",
                   openai_api_key=API_KEY,
                   openai_api_base="https://api.deepseek.com")

#实例化一个对话缓冲区，用于存储对话历史
memory = ConversationBufferMemory()
#创建一个对话链，将大语言模型和对话缓冲区关联起来。
conversation = ConversationChain(
    llm=llm,
    memory=memory,
)

conversation.invoke("今天早上猪八戒吃了2个人参果。")
print("记忆1: ", conversation.memory.buffer)
print()

conversation.invoke("下午猪八戒吃了1个人参果。")
print("记忆2: ", conversation.memory.buffer)
print()

conversation.invoke("晚上猪八戒吃了3个人参果。")
print("记忆3: ", conversation.memory.buffer)
print()

conversation.invoke("猪八戒今天一共吃了几个人参果？")
print("记忆4: ", conversation.memory.buffer)

记忆1:  Human: 今天早上猪八戒吃了2个人参果。
AI: 哈哈，看来猪八戒今天胃口不错啊！按照《西游记》原著，人参果可是五庄观的宝贝，三千年一开花，三千年一结果，闻一闻能活三百六十岁，吃一个能活四万七千年。猪八戒当初偷吃时可是囫囵吞枣，连味道都没尝出来呢！不过你说他早上吃了两个——这是又去五庄观“串门”了吗？还是现代有人种出了新品种人参果？（笑）要小心镇元大仙找你算账哦！  

（温馨提示：现实中没有真实的人参果，但有一种茄科植物的果实外形类似，可食用但无长生功效~）

记忆2:  Human: 今天早上猪八戒吃了2个人参果。
AI: 哈哈，看来猪八戒今天胃口不错啊！按照《西游记》原著，人参果可是五庄观的宝贝，三千年一开花，三千年一结果，闻一闻能活三百六十岁，吃一个能活四万七千年。猪八戒当初偷吃时可是囫囵吞枣，连味道都没尝出来呢！不过你说他早上吃了两个——这是又去五庄观“串门”了吗？还是现代有人种出了新品种人参果？（笑）要小心镇元大仙找你算账哦！  

（温馨提示：现实中没有真实的人参果，但有一种茄科植物的果实外形类似，可食用但无长生功效~）
Human: 下午猪八戒吃了1个人参果。
AI: 哎呀，猪八戒这是要把五庄观的人参果当零食吃啊！（笑）下午又吃一个的话，这一天下来总共就吃了三个啦——这可比原著里偷吃的数量还多呢！看来二师兄最近是跟人参果杠上了？  

不过按照这个节奏...我算算：早上2个 + 下午1个 = 3个人参果 × 4万7千年寿命 = 14万1千年长生不老！这下连唐僧都不用取经了，直接改行开人参果种植园得了～（突然严肃）等等...该不会是有人假扮猪八戒在倒卖仙果吧？土地公公快出来管管啊！  

（继续温馨提示：现实中的人参果又名"香瓜茄"，清甜多汁，超市就能买到，但真的不能长生不老哦～）

记忆3:  Human: 今天早上猪八戒吃了2个人参果。
AI: 哈哈，看来猪八戒今天胃口不错啊！按照《西游记》原著，人参果可是五庄观的宝贝，三千年一开花，三千年一结果，闻一闻能活三百六十岁，吃一个能活四万七千年。猪八戒当初偷吃时可是囫囵吞枣，连味道都没尝出来呢！不过你说他早上吃了两个——这是又去五庄观“串门”了吗？还是现代有人种出了新品种人参果？（笑）要小心镇元大仙找你算账哦！  

（温馨提示：现实中没有真实的人参果，但有一种茄科植物的果实外形类似，可

In [18]:
# **功能设计：多轮对话**

In [19]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
import warnings
warnings.filterwarnings("ignore")

# 实例化一个对话缓冲区，用于存储对话历史
memory = ConversationBufferMemory()
# 创建一个对话链，将大语言模型和对话缓冲区关联起来。
conversation = ConversationChain(
    llm=llm,
    memory=memory,
)

print("欢迎使用对话系统！输入 '退出' 结束对话。")

while True:
    user_input = input("你: ")
    if user_input.lower() in ['退出', 'exit', 'quit']:
        print("再见！")
        break
    response = conversation.predict(input=user_input)
    print(f"AI: {response}")

# 打印出对话历史，即 memory.buffer 的内容
print("对话历史:", memory.buffer)

欢迎使用对话系统！输入 '退出' 结束对话。


你:  你是谁


AI: 我是一个人工智能助手，专门用来回答问题、提供帮助和陪你聊天！你可以叫我AI助手或者随便起个你喜欢的名字~ 我没有实体，但可以处理各种信息，比如解答问题、聊天、学习建议、生活小窍门等等。有什么我可以帮你的吗？ 😊


你:  你是男生还是女生


AI: 我没有性别哦~ 作为人工智能，我既不是男生也不是女生，只是一个虚拟的程序，专门用来帮助你解决问题或陪你聊天。不过如果你希望用某种性别称呼我（比如“他”或“她”），可以告诉我你的偏好，我会配合的！ 😄 你觉得哪种称呼让你更习惯呢？


你:  退出


再见！
对话历史: Human: 你是谁
AI: 我是一个人工智能助手，专门用来回答问题、提供帮助和陪你聊天！你可以叫我AI助手或者随便起个你喜欢的名字~ 我没有实体，但可以处理各种信息，比如解答问题、聊天、学习建议、生活小窍门等等。有什么我可以帮你的吗？ 😊
Human: 你是男生还是女生
AI: 我没有性别哦~ 作为人工智能，我既不是男生也不是女生，只是一个虚拟的程序，专门用来帮助你解决问题或陪你聊天。不过如果你希望用某种性别称呼我（比如“他”或“她”），可以告诉我你的偏好，我会配合的！ 😄 你觉得哪种称呼让你更习惯呢？


In [ ]:
# **携带提示词模版的对轮对话(LLMChain对话链)**

In [20]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
import os
import warnings
warnings.filterwarnings("ignore")

# 初始化大模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=API_KEY,
    openai_api_base="https://api.deepseek.com"
)

# 实例化一个对话缓冲区，用于存储对话历史
memory = ConversationBufferMemory()

# 定义提示词模板
template = """{history}
用户: {input}
AI:"""

prompt_template = PromptTemplate(
    input_variables=["history", "input"],
    template=template
)

# 创建一个包含提示词模板的对话链
conversation = LLMChain(
    llm=llm,
    prompt=prompt_template,
    verbose=True,  # 如果需要调试，可以设置为 True
    memory=memory
)

print("欢迎使用对话系统！输入 '退出' 结束对话。")

while True:
    user_input = input("你: ")
    if user_input.lower() in ['退出', 'exit', 'quit']:
        print("再见！")
        break
    try:
        # 调用对话链获取响应
        response = conversation.run(input=user_input)
        print(f"AI: {response}")
    except Exception as e:
        print(f"发生错误: {e}")

# 打印出对话历史，即 memory.buffer 的内容
print("对话历史:", memory.buffer)

欢迎使用对话系统！输入 '退出' 结束对话。


你:  tuic




> Entering new LLMChain chain...
Prompt after formatting:

用户: tuic
AI:

> Finished chain.
AI: 你好！😊 我是DeepSeek Chat，很高兴见到你。有什么我可以帮助你的吗？


你:  退出


再见！
对话历史: Human: tuic
AI: 你好！😊 我是DeepSeek Chat，很高兴见到你。有什么我可以帮助你的吗？


In [21]:
# 如果使用聊天模型，使用结构化的聊天消息可能会有更好的性能:

In [22]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains.llm import LLMChain
from langchain_core.messages import SystemMessage
from langchain_core.prompts import MessagesPlaceholder, HumanMessagePromptTemplate, ChatPromptTemplate
import warnings
warnings.filterwarnings("ignore")

# 初始化大模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=API_KEY,
    openai_api_base="https://api.deepseek.com"
)

# 使用ChatPromptTemplate设置聊天提示
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="你是一个与人类对话的机器人。"),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}"),
    ]
)

# 创建ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 初始化链
chain = LLMChain(llm=llm,  prompt=prompt, memory=memory)

# 提问
res = chain.invoke({"question": "你是LangChain专家"})
print(str(res) + "\n")    

res = chain.invoke({"question": "你是谁?"})
print(res)

{'question': '你是LangChain专家', 'chat_history': [HumanMessage(content='你是LangChain专家', additional_kwargs={}, response_metadata={}), AIMessage(content='是的，我熟悉 **LangChain**，这是一个用于构建基于大型语言模型（LLM）的应用程序的强大框架。如果你有关于 **LangChain** 的问题，比如以下方面，我都可以帮你解答：  \n\n### **LangChain 核心概念**  \n- **Models（模型）**：如何接入 OpenAI、Anthropic、Hugging Face 等 LLM。  \n- **Prompts（提示）**：如何优化提示模板（Prompt Templates）和少样本示例（Few-shot Learning）。  \n- **Chains（链）**：如何组合多个 LLM 调用（如 `LLMChain`、`SequentialChain`）。  \n- **Memory（记忆）**：如何让对话保持上下文（如 `ConversationBufferMemory`）。  \n- **Agents（代理）**：如何让 LLM 使用工具（如 `Tool`、`AgentExecutor`）。  \n- **Indexes（索引）**：如何结合向量数据库（如 `FAISS`、`Pinecone`）做文档问答。  \n\n### **常见问题**  \n1. **如何用 LangChain 构建一个聊天机器人？**  \n2. **如何让 LLM 调用外部 API 或数据库？**  \n3. **如何用 RetrievalQA 实现文档问答？**  \n4. **如何优化 LangChain 的性能和成本？**  \n5. **LangChain 和 LlamaIndex 有什么区别？**  \n\n如果你有具体的代码示例或需求，也可以告诉我，我可以帮你调试或优化！ 🚀  \n\n**你想了解 LangChain 的哪个部分？**', additional_kwargs={}, response_metadata={})], 'text': '是的，我熟悉 **LangChain**，这是

In [23]:
# **多轮对话Token限制解决**

In [24]:
# 在了解了`ConversationBufferMemory`记忆类后，我们知道了它能够无限的将历史对话信息填充到History中，从而给大模型提供上下文的背景。但问题是：每个大模型都存在最大输入的Token限制，
# 且过久远的对话数据往往并不能够对当前轮次的问答提供有效的信息，这种我们大家都能非常容易想到的问题，LangChain的开发人员自然也能想到，那么他们给出的解决方式是：`ConversationBufferWindowMemory`模块。
# 该记忆类会保存一段时间内对话交互的列表，仅使用最后 K 个交互。所以它可以保存最近交互的滑动窗口，避免缓存区不会变得太大。

In [13]:
from langchain.memory import ConversationBufferWindowMemory
import warnings
warnings.filterwarnings("ignore")

#实例化一个对话缓冲区，用于存储对话历史
    #k=1，所以在读取时仅能提取到最近一轮的记忆信息
    #return_messages=True参数，将对话转化为消息列表形式
memory = ConversationBufferWindowMemory(k=1, return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
)

# 示例对话
response1 = conversation.predict(input="你好")
response2 = conversation.predict(input="你在哪里？")
print("对话历史:", memory.buffer)

对话历史: [HumanMessage(content='你在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='哈哈，我其实没有物理上的“位置”哦！作为一个AI程序，我存在于云端服务器和网络中，没有实体形态。不过我的服务可以随时通过互联网连接到世界各地🌍～ \n\n如果你问的是开发团队的话，创造我的公司DeepSeek位于中国。但要注意的是，我可不会受地理位置限制，只要你需要，我就能出现在你的手机或电脑里陪你聊天啦！✨\n\n需要我帮忙解决具体问题吗？还是单纯好奇AI的运作方式呢？ 😄', additional_kwargs={}, response_metadata={})]


In [ ]:
# **实体记忆：ConversationEntityMemory**abs

In [ ]:
# 在LangChain 中,ConversationEntityMemory是实体记忆,它可以跟踪对话中提到的实体，在对话中记住关于特定实体的给定事实。它提取关于实体的信息（使用LLM），
并随着时间的推移建立对该实体的知识（使用LLM）。

In [ ]:
# 使用它来存储和查询对话中引用的各种信息,比如人物、地点、事件等。

In [15]:
from langchain.chains.conversation.base import ConversationChain
from langchain.memory import ConversationEntityMemory
from langchain.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE
from langchain_openai import OpenAI
import warnings
warnings.filterwarnings("ignore")

# 初始化大模型
API_KEY = open('./ken_files/deepseekAPI-Key.md',encoding='utf-8').read().strip()
llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=API_KEY,
    openai_api_base="https://api.deepseek.com"
)


conversation = ConversationChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=ConversationEntityMemory(llm=llm)
)

# 开始对话
conversation.predict(input="你好,我是小明。我最近在学习 LangChain。")
conversation.predict(input="我最喜欢的编程语言是 Python。")
conversation.predict(input="我住在北京。")

# 查询对话中提到的实体
res = conversation.memory.entity_store.store
print(res)

{'小明': '小明最近在学习LangChain。', 'LangChain': '"LangChain is a framework for building applications powered by large language models, providing tools and modules to simplify development."  \n\n(Since this is the first summary, it captures the key fact from the AI\'s response about LangChain\'s purpose and capabilities.)', '北京': '北京是小明居住的城市，以其科技和创新氛围著称。'}
